# 05 — Knowledge Base RAG dari Dokumen Panduan Penyakit Cabai

Membangun "perpustakaan" yang nanti dibaca agent sebelum menjawab petani:

1. **Baca** semua dokumen di `docs/` (PDF, Markdown), buang watermark/header berulang.
2. **Potong** menjadi chunk ±900 karakter, beri **tag penyakit** (sesuai 6 kelas model).
3. **Embedding** dengan `multilingual-e5-small` (paham Bahasa Indonesia), simpan ke `rag_index/`.
4. **Evaluasi retrieval** dengan 30 pertanyaan uji: model embedding vs pembanding TF-IDF, dengan dan tanpa petunjuk kelas dari model CNN.

Jalankan di laptop (CPU cukup). Saat pertama kali jalan, model embedding (±470 MB) diunduh otomatis.

## 1. Konfigurasi

In [4]:
import sys, json, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

DOCS_DIR   = PROJECT_ROOT / "docs"
INDEX_DIR  = PROJECT_ROOT / "rag_index"          # hasil index; ikut di-upload ke Hugging Face Space nanti
OUT_DIR    = PROJECT_ROOT / "reports" / "rag"
EMBEDDER   = "e5"                                 # "e5" (utama) atau "tfidf" (tanpa unduh model)
CHUNK_SIZE, CHUNK_OVERLAP = 900, 150
TOP_K = 3

OUT_DIR.mkdir(parents=True, exist_ok=True)
files = sorted(p.name for p in DOCS_DIR.iterdir() if p.suffix.lower() in {".pdf", ".md", ".txt"})
print(f"{len(files)} dokumen di {DOCS_DIR}:")
for f in files:
    print("  -", f)
assert files, "Folder docs/ masih kosong."

6 dokumen di c:\laragon\www\final-project\dokter-tani\docs:
  - 14bookcabe_watermark.pdf
  - Berita BBRMP Babel - PENERAPAN SNI 4480_2016 SEBAGAI PEDOMAN PENGENDALIAN PENYAKIT PADA TANAMAN CABAI.pdf
  - Penyakit Bercak Daun Serkospora (Cercospora capsici) pada Tanaman Cabai _ Dinas Pertanian, Ketahanan Pangan Dan Perikanan.pdf
  - SOP-Cabai-Merah-2014.pdf
  - Teknik-Pengendalian-Penyakit-Kuning-Pada-Tanaman-Cabai_watermark.pdf
  - ringkasan_dokter_tani.md


In [5]:
import numpy as np
import pandas as pd
from src.rag.knowledge import (load_documents, chunk_documents, source_info, KnowledgeBase,
                               E5Embedder, TfidfEmbedder, DISEASE_KEYWORDS)
from src.model.predictor import display_name

pd.set_option("display.max_colwidth", 120)
CLASSES = list(DISEASE_KEYWORDS)

## 2. Inventaris dokumen
Periksa apakah teks setiap dokumen terbaca. PDF hasil *scan* (gambar) hampir tidak punya teks, jadi perlu OCR atau diganti.

In [6]:
pages = load_documents(DOCS_DIR)
inv = (pd.DataFrame(pages).assign(karakter=lambda d: d["text"].str.len())
         .groupby("source").agg(halaman_berteks=("text", "size"), karakter=("karakter", "sum"))
         .reindex(files).fillna(0).astype(int))
inv["karakter_per_halaman"] = (inv["karakter"] / inv["halaman_berteks"].clip(lower=1)).round()
inv["judul"] = [source_info(s)["judul"] for s in inv.index]
inv["status"] = np.where(inv["karakter"] == 0, "✗ tidak ada teks (PDF scan?)", np.where(inv["karakter_per_halaman"] < 150, "⚠ teks sedikit", "OK"))
inv.to_csv(OUT_DIR / "inventaris_dokumen.csv")
inv

,halaman_berteks,karakter,karakter_per_halaman,judul,status
source,,,,,
14bookcabe_watermark.pdf,26,23183,892.0,Hama dan Penyakit pada Tanaman Cabai serta Pengendaliannya,OK
Berita BBRMP Babel - PENERAPAN SNI 4480_2016 SEBAGAI PEDOMAN PENGENDALIAN PENYAKIT PADA TANAMAN CABAI.pdf,11,19418,1765.0,Penerapan SNI 4480:2016 sebagai Pedoman Pengendalian Penyakit pada Tanaman Cabai,OK
"Penyakit Bercak Daun Serkospora (Cercospora capsici) pada Tanaman Cabai _ Dinas Pertanian, Ketahanan Pangan Dan Perikanan.pdf",2,4452,2226.0,Penyakit Bercak Daun Serkospora pada Tanaman Cabai,OK
SOP-Cabai-Merah-2014.pdf,75,69904,932.0,SOP Budidaya Cabai Merah,OK
Teknik-Pengendalian-Penyakit-Kuning-Pada-Tanaman-Cabai_watermark.pdf,31,26314,849.0,Teknik Pengendalian Penyakit Kuning pada Tanaman Cabai,OK
ringkasan_dokter_tani.md,5,6100,1220.0,Ringkasan Dokter Tani (disusun dari sumber resmi),OK


## 3. Chunking dan cakupan per kelas
Setiap chunk diberi tag kelas berdasarkan kata kunci (misalnya *xanthomonas* → Bercak bakteri). Kelas dengan sedikit chunk = jawaban bot untuk kelas itu akan tipis.

In [7]:
chunks = chunk_documents(pages, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
cdf = pd.DataFrame(chunks)
print(f"{len(cdf)} chunk | panjang rata-rata {cdf['text'].str.len().mean():.0f} karakter | "
      f"tanpa tag kelas: {(cdf['tags'].str.len() == 0).sum()}")

cov = pd.DataFrame({c: cdf.groupby("source")["tags"].apply(lambda s: sum(c in t for t in s)) for c in CLASSES})
cov.columns = [display_name(c) for c in CLASSES]
cov.loc["TOTAL"] = cov.sum()
cov.to_csv(OUT_DIR / "cakupan_per_kelas.csv")
cov

236 chunk | panjang rata-rata 678 karakter | tanpa tag kelas: 87


,Bercak Bakteri,Bercak Daun Cercospora,Virus Keriting Daun,Embun Tepung,Kekurangan Nutrisi,Daun Sehat
source,,,,,,
14bookcabe_watermark.pdf,0,3,12,0,0,11
Berita BBRMP Babel - PENERAPAN SNI 4480_2016 SEBAGAI PEDOMAN PENGENDALIAN PENYAKIT PADA TANAMAN CABAI.pdf,2,1,4,1,3,15
"Penyakit Bercak Daun Serkospora (Cercospora capsici) pada Tanaman Cabai _ Dinas Pertanian, Ketahanan Pangan Dan Perikanan.pdf",0,3,0,0,0,2
SOP-Cabai-Merah-2014.pdf,0,1,8,0,9,60
Teknik-Pengendalian-Penyakit-Kuning-Pada-Tanaman-Cabai_watermark.pdf,0,1,24,0,1,15
ringkasan_dokter_tani.md,1,1,3,4,4,6
TOTAL,3,10,51,5,17,109


In [8]:
# Contoh chunk untuk tiap kelas: cek apakah isinya memang relevan
for c in CLASSES:
    sub = cdf[cdf["tags"].apply(lambda t: c in t)]
    if len(sub):
        r = sub.iloc[0]
        print(f"■ {display_name(c)}  ←  {KnowledgeBase.cite(r)}\n{r['text'][:350]}...\n")
    else:
        print(f"■ {display_name(c)}: TIDAK ADA chunk. Tambahkan dokumen untuk kelas ini.\n")

■ Bercak Bakteri  ←  Penerapan SNI 4480:2016 sebagai Pedoman Pengendalian Penyakit pada Tanaman Cabai (BSIP/BRMP Bangka Belitung, 2024) hal. 5
g atau klorosis. Pengendalian dilakukan dengan membuang semaian yang
terinfeksi dan mengendalikan vektornya (kutu daun) dengan insektisida yang tepat.
3.    Penyakit Fase Vegetatif dan Generatif
a.    Bakteri
Penyakit ini dinamakan Bercak bakteri yang disebabkan oleh patogen Xanthomonas
campestris p.v. vesicatoria (Xcv). Gejala yang ditibulkan adal...

■ Bercak Daun Cercospora  ←  Hama dan Penyakit pada Tanaman Cabai serta Pengendaliannya (BPTP Jambi, 2014) hal. 5
ii
DAFTAR ISI
Hal 
Kata Pengantar ........................................................... i 
Daftar Isi .................................................................... ii 
Daftar Gambar ............................................................ iii 
I. Hama Penting pada Tanaman Cabai ..................... 1 
1. Thrips (Thrips parvispinus...

■ Virus Keriting Daun  ←  Hama da

## 4. Membangun index
Index utama (`EMBEDDER`) disimpan ke `rag_index/`. TF-IDF dibangun juga sebagai pembanding di evaluasi.

In [9]:
t0 = time.time()
emb_main = E5Embedder() if EMBEDDER == "e5" else TfidfEmbedder()
kb = KnowledgeBase.build(DOCS_DIR, emb_main, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
kb.save(INDEX_DIR)
print(f"Index {emb_main.name}: {len(kb.chunks)} chunk, dimensi {kb.emb.shape[1]} | {time.time() - t0:.0f} dtk | disimpan ke {INDEX_DIR}")

kb_tfidf = kb if EMBEDDER == "tfidf" else KnowledgeBase.build(DOCS_DIR, TfidfEmbedder(), size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

Index intfloat/multilingual-e5-small: 236 chunk, dimensi 384 | 84 dtk | disimpan ke c:\laragon\www\final-project\dokter-tani\rag_index


## 5. Evaluasi retrieval: 30 pertanyaan uji

Setiap pertanyaan punya **kelas yang diharapkan**. Sebuah hasil dianggap **relevan** bila chunk yang ditemukan bertag kelas tersebut.

- **Hit@1 / Hit@3:** persentase pertanyaan yang chunk relevannya muncul di urutan 1 / 3 teratas.
- **MRR:** rata-rata 1/peringkat chunk relevan pertama (1 = selalu di urutan pertama).
- **Dengan petunjuk kelas:** agent tahu hasil diagnosis CNN, sehingga chunk kelas itu diberi bonus skor (seperti di bot nanti).

Catatan: relevansi berbasis tag kata kunci adalah perkiraan. Periksa juga beberapa hasil secara manual di bagian 6.

In [10]:
EVAL = [
    ("Apa gejala bercak daun cercospora pada cabai?", "Cercospora_Leaf_Spot"),
    ("Bagaimana cara mengendalikan bercak daun serkospora?", "Cercospora_Leaf_Spot"),
    ("Kondisi cuaca apa yang memicu bercak daun cercospora?", "Cercospora_Leaf_Spot"),
    ("Bagaimana jamur penyebab bercak daun menyebar ke tanaman lain?", "Cercospora_Leaf_Spot"),
    ("Daun cabai ada bercak bulat dengan pusat abu-abu dan tepi coklat, penyakit apa?", "Cercospora_Leaf_Spot"),
    ("Apa penyebab daun cabai keriting dan menguning?", "Curl_Virus"),
    ("Serangga apa yang menularkan virus kuning pada cabai?", "Curl_Virus"),
    ("Bagaimana mencegah penyakit virus kuning keriting?", "Curl_Virus"),
    ("Apakah tanaman yang terkena virus kuning bisa disembuhkan?", "Curl_Virus"),
    ("Bagaimana cara mengendalikan kutu kebul?", "Curl_Virus"),
    ("Apa gejala bercak bakteri pada daun cabai?", "Bacterial_Spot"),
    ("Bagaimana mengendalikan penyakit bercak bakteri?", "Bacterial_Spot"),
    ("Bagaimana perlakuan benih untuk mencegah bakteri Xanthomonas?", "Bacterial_Spot"),
    ("Seperti apa gejala bercak bakteri pada buah cabai?", "Bacterial_Spot"),
    ("Ada bercak coklat kehitaman di daun dan ranting cabai, apa penyebabnya?", "Bacterial_Spot"),
    ("Apa gejala embun tepung pada cabai?", "Powdery_Mildew"),
    ("Jamur apa penyebab penyakit embun tepung?", "Powdery_Mildew"),
    ("Bagaimana cara mengendalikan embun tepung?", "Powdery_Mildew"),
    ("Pada kondisi apa penyakit embun tepung berkembang?", "Powdery_Mildew"),
    ("Kenapa ada lapisan putih seperti tepung di bawah daun cabai?", "Powdery_Mildew"),
    ("Apa tanda tanaman cabai kekurangan nitrogen?", "Nutrition_Deficiency"),
    ("Bagaimana gejala kekurangan kalium pada daun cabai?", "Nutrition_Deficiency"),
    ("Bagaimana membedakan kekurangan hara dengan penyakit virus?", "Nutrition_Deficiency"),
    ("Daun muda menguning tapi tulang daun tetap hijau, kekurangan unsur apa?", "Nutrition_Deficiency"),
    ("Bagaimana cara mengatasi kekurangan unsur hara pada cabai?", "Nutrition_Deficiency"),
    ("Bagaimana cara pemupukan cabai yang baik?", "Healthy_Leaf"),
    ("Bagaimana merawat tanaman cabai agar tetap sehat?", "Healthy_Leaf"),
    ("Apa manfaat mulsa plastik perak pada budidaya cabai?", "Healthy_Leaf"),
    ("Kapan waktu pemupukan susulan tanaman cabai?", "Healthy_Leaf"),
    ("Bagaimana menyiapkan persemaian cabai yang sehat?", "Healthy_Leaf"),
]

def evaluate(kb, hint, k=TOP_K):
    rows = []
    for q, c in EVAL:
        res = kb.search(q, k=10, disease=c if hint else None)
        ranks = [i + 1 for i, r in enumerate(res) if c in r["tags"]]
        first = ranks[0] if ranks else None
        rows.append({"pertanyaan": q, "kelas": display_name(c), "peringkat_relevan": first,
                     "hit1": first == 1, f"hit{k}": first is not None and first <= k,
                     "rr": 1 / first if first else 0.0, "sumber_teratas": KnowledgeBase.cite(res[0])})
    return pd.DataFrame(rows)

MAIN = "TF-IDF" if EMBEDDER == "tfidf" else emb_main.name.split("/")[-1]
retrievers = [("TF-IDF", kb)] if EMBEDDER == "tfidf" else [("TF-IDF", kb_tfidf), (MAIN, kb)]
results, summary = {}, []
for name, base in retrievers:
    for hint in (False, True):
        r = evaluate(base, hint)
        results[(name, hint)] = r
        summary.append({"retriever": name, "petunjuk_kelas": "ya" if hint else "tidak",
                        "Hit@1": r["hit1"].mean().round(3), f"Hit@{TOP_K}": r[f"hit{TOP_K}"].mean().round(3),
                        "MRR": r["rr"].mean().round(3)})
summary = pd.DataFrame(summary)
summary.to_csv(OUT_DIR / "evaluasi_retrieval.csv", index=False)
summary

,retriever,petunjuk_kelas,Hit@1,Hit@3,MRR
0,TF-IDF,tidak,0.967,1.0,0.983
1,TF-IDF,ya,1.000,1.0,1.000
2,multilingual-e5-small,tidak,0.867,1.0,0.933
3,multilingual-e5-small,ya,1.000,1.0,1.000


In [11]:
# Per kelas (retriever utama, dengan petunjuk kelas)
main = results[(MAIN, True)]
per_kelas = main.groupby("kelas")[["hit1", f"hit{TOP_K}", "rr"]].mean().round(2).rename(columns={"rr": "MRR"})
per_kelas

,hit1,hit3,MRR
kelas,,,
Bercak Bakteri,1.0,1.0,1.0
Bercak Daun Cercospora,1.0,1.0,1.0
Daun Sehat,1.0,1.0,1.0
Embun Tepung,1.0,1.0,1.0
Kekurangan Nutrisi,1.0,1.0,1.0
Virus Keriting Daun,1.0,1.0,1.0


## 6. Cek manual
Pertanyaan yang gagal (chunk relevan tidak masuk 3 besar), lalu contoh hasil pencarian lengkap.

In [12]:
fails = main[~main[f"hit{TOP_K}"]]
print(f"Gagal: {len(fails)} dari {len(main)} pertanyaan")
fails[["pertanyaan", "kelas", "peringkat_relevan", "sumber_teratas"]]

Gagal: 0 dari 30 pertanyaan


,pertanyaan,kelas,peringkat_relevan,sumber_teratas


In [13]:
def tanya(q, disease=None, k=TOP_K):
    print(f"❓ {q}" + (f"   [petunjuk: {display_name(disease)}]" if disease else ""))
    for i, r in enumerate(kb.search(q, k=k, disease=disease), 1):
        print(f"\n  {i}. skor {r['score']:.3f} · {KnowledgeBase.cite(r)} · tag: {', '.join(display_name(t) for t in r['tags']) or '-'}")
        print("     " + r["text"][:300].replace("\n", " ") + "...")
    print("\n" + "-" * 100)

tanya("Bagaimana cara mengendalikan embun tepung?", "Powdery_Mildew")
tanya("Daun cabai saya keriting dan kuning, harus bagaimana?", "Curl_Virus")
tanya("Pupuk apa yang perlu ditambah kalau daun tua menguning?")

❓ Bagaimana cara mengendalikan embun tepung?   [petunjuk: Embun Tepung]

  1. skor 0.946 · Ringkasan Dokter Tani (disusun dari sumber resmi) (Tim Dokter Tani, 2026) · tag: Embun Tepung, Daun Sehat
     rsihan lahan dan lakukan pemupukan berimbang. 4. Bila serangan meluas, gunakan fungisida yang terdaftar untuk embun tepung, sesuai anjuran pada label. Sumber: BSIP/BRMP Bangka Belitung (2024), *Penerapan SNI 4480:2016*; Politeknik Pertanian Negeri Kupang, *Penyakit Embun Tepung*....

  2. skor 0.945 · Ringkasan Dokter Tani (disusun dari sumber resmi) (Tim Dokter Tani, 2026) · tag: Embun Tepung, Daun Sehat
     ## Embun tepung (powdery mildew) pada cabai **Penyebab.** Jamur *Leveillula taurica*, yang menyerang cabai dan terung. Spora jamur menyebar terbawa angin. **Gejala.** Bercak kekuningan pada permukaan atas daun. Pada daun, terutama permukaan bawah, terlihat lapisan putih keabu-abuan seperti tepung. D...

  3. skor 0.943 · Penerapan SNI 4480:2016 sebagai Pedoman Pengendalian Penyakit

## 7. Ringkasan untuk `CATATAN.md`

In [14]:
best = summary.sort_values(["MRR", "Hit@1"], ascending=False).iloc[0]
print(f"| {time.strftime('%Y-%m-%d')} | Knowledge base RAG: {len(inv)} dokumen, {len(kb.chunks)} chunk, {emb_main.name} | "
      f"terbaik: {best['retriever']} (petunjuk kelas: {best['petunjuk_kelas']}) Hit@1 {best['Hit@1']:.2f}, "
      f"Hit@{TOP_K} {best[f'Hit@{TOP_K}']:.2f}, MRR {best['MRR']:.2f} | "
      f"Kelas terlemah: {per_kelas['MRR'].idxmin()} (MRR {per_kelas['MRR'].min():.2f}) |")

| 2026-10-18 | Knowledge base RAG: 6 dokumen, 236 chunk, intfloat/multilingual-e5-small | terbaik: TF-IDF (petunjuk kelas: ya) Hit@1 1.00, Hit@3 1.00, MRR 1.00 | Kelas terlemah: Bercak Bakteri (MRR 1.00) |


### Yang perlu diamati
- **Inventaris:** ada dokumen berstatus ⚠? Kemungkinan PDF hasil scan; ganti atau jalankan OCR.
- **Cakupan:** kelas dengan chunk paling sedikit adalah kandidat untuk ditambah dokumennya.
- **Evaluasi:** embedding e5 seharusnya mengalahkan TF-IDF, dan petunjuk kelas dari CNN seharusnya menaikkan Hit@1. Selisih ini adalah bukti bagus untuk slide metodologi.

**Berikutnya:** `06_agent`, yaitu LLM yang memakai knowledge base ini (dan cuaca, opsional) untuk menyusun jawaban bot.